# Stage 4b - Label

Codebook: `src/krs/label/codebook.md` (read it before labelling). Settings: `configs/label.yaml`.

**Order:** 1 gold ids → 2 rater sheets → 3 training pool → 4 LLM weak labels (Batch API) → 5 noise check → 6 agreement (κ) → 7 adjudication → 8 final gold.

Only step 4 costs money. It needs `ANTHROPIC_API_KEY` set in Colab secrets.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[label]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

## 1-2. Gold set and rater sheets

Each rater gets the same sheet and labels it **alone**: put `x` in every `k:` (knowledge type) and `d:` (domain) column that applies. Do not share sheets until both are done.

In [ ]:
from krs.config import get_paths, load_config
from krs.label import sampling, gold

cfg = load_config("label")
L = get_paths().labels
gold_ids = sampling.select_gold(cfg["gold"]["n_segments"], seed=cfg["gold"]["seed"])
segs = sampling.load_segments(gold_ids)
for rater in ("rater1", "rater2"):
    gold.export_rater_sheet([segs[i] for i in gold_ids], L / f"{rater}.csv")
print(len(gold_ids), "gold segments; sheets in", L)

## 3. Training pool

Excludes every conversation that has a gold segment, so no test conversation can leak into training.

In [ ]:
pool_ids = sampling.select_pool(cfg["pool"]["n_segments"], gold_ids, seed=cfg["pool"]["seed"], allocation=cfg["pool"]["allocation"])
len(pool_ids)

## 4. LLM weak labels (costs money)

Check the estimate first. Batch mode is 50% cheaper; most batches finish within an hour. Re-running `collect_batch` is safe.

In [ ]:
import statistics
from krs.label import llm_labeller as LL

psegs = sampling.load_segments(pool_ids)
cb = LL.load_codebook()
seg_tok = statistics.mean(len(LL.segment_prompt(s)) for s in psegs.values()) / 4
cb_tok = (len(cb) + len(LL.INSTRUCTIONS)) / 4
p = cfg["llm"]["prices"]
for out_tok in (200, 600):
    est = (seg_tok * p["input"] + cb_tok * p["cache_read"] + out_tok * p["output"]) / 1e6 * 0.5 * len(psegs)
    print(f"~${est:,.0f} if ~{out_tok} output tokens per segment ({len(psegs)} segments, {cfg['llm']['model']})")

In [ ]:
# Submit only after checking the estimate above
state = L / "weak_batch_state.json"
LL.submit_batch(psegs.values(), state, cfg["llm"], done=LL.done_ids(L / "weak.jsonl"))

In [ ]:
LL.batch_status(state)   # re-run until every batch says 'ended'

In [ ]:
report = LL.collect_batch(state, L / "weak.jsonl", cfg["llm"], codebook_version=cfg["codebook_version"])
report   # failures are in weak_errors.jsonl; re-submit them later

## 5. Noise check of LLM labels

Mark `types_correct` and `domains_correct` with `y` or `n` for each of the 100 rows.

In [ ]:
from krs.label import noise

noise.export_noise_sample(L / "weak.jsonl", L / "noise_check.csv", n=cfg["noise_check_sample"])

In [ ]:
noise.noise_rate(L / "noise_check.csv")   # after filling the sheet

## 6-8. Agreement, adjudication, final gold

After both raters return their sheets. If κ is below 0.6, revise the codebook (bump to v2), relabel once, and report both values.

In [ ]:
from krs.label import agreement
from krs.schema import LabelSource

r1, p1 = gold.import_rater_sheet(L / "rater1.csv", LabelSource.RATER_1, "rater1")
r2, p2 = gold.import_rater_sheet(L / "rater2.csv", LabelSource.RATER_2, "rater2")
print("problems:", p1 + p2)   # fix these rows in the sheets first
agreement.agreement(r1, r2, cfg["gold"]["kappa_threshold"])

In [ ]:
agreement.export_adjudication_sheet(r1, r2, segs, L / "adjudicate.csv")   # third member fills this

In [ ]:
from krs.io import write_jsonl

final, problems = agreement.merge_gold(r1, r2, L / "adjudicate.csv")
print(len(final), "gold labels; problems:", problems)
write_jsonl(L / "gold.jsonl", final)

## Teacher run T (Stage 7, listed here because it uses the same labeller)

Labels the gold set with the LLM, one request at a time, to measure the teacher's F1 ceiling, cost and latency per 1,000 segments:

```python
LL.label_sync(segs.values(), get_paths().labels / "teacher.jsonl", {**cfg["llm"], "mode": "sync"})
```